In [1]:
import pandas as pd
import joblib

In [2]:
from sklearn.metrics import roc_auc_score, accuracy_score

In [3]:
X_val = pd.read_parquet('../data/X_val.parquet')
X_test = pd.read_parquet('../data/X_test.parquet')
y_val = pd.read_parquet('../data/y_val.parquet').squeeze()
y_test = pd.read_parquet('../data/y_test.parquet').squeeze()

In [4]:
X_val.columns.tolist()

['DEPARTURE_HOUR',
 'wind_gusts',
 'precipitation',
 'cloud_cover_low',
 'weather_code',
 'rain',
 'temperature',
 'wind_speed',
 'snowfall',
 'ORIGIN_HOURLY_FLIGHTS',
 'DAY_OF_WEEK',
 'MONTH',
 'IS_HOLIDAY_PERIOD',
 'DISTANCE',
 'CRS_ELAPSED_TIME',
 'ORIGIN_STATE_ABR',
 'DEST_STATE_ABR',
 'ROUTE_HOURLY_FLIGHTS',
 'OP_UNIQUE_CARRIER',
 'DEST_HOURLY_FLIGHTS',
 'CARRIER_DELAY_RATE',
 'ORIGIN_DELAY_RATE',
 'ROUTE_DELAY_RATE',
 'DEST_DELAY_RATE',
 'ORIGIN_ENCODED',
 'DEST_ENCODED',
 'ROUTE_ENCODED']

In [5]:
final_model = joblib.load('../models/best_model.pkl')

In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

### 1. AUC-ROC

In [7]:
y_val_pred_prob = final_model.predict_proba(X_val)[:, 1]
y_val_auc = roc_auc_score(y_val, y_val_pred_prob)
print(f'{round(y_val_auc * 100, 2)}%')

74.75%


In [8]:
y_test_pred_prob = final_model.predict_proba(X_test)[:, 1]
y_test_auc = roc_auc_score(y_test, y_test_pred_prob)
print(f'{round(y_test_auc * 100, 2)}%')

74.62%


### 2. Applying Threshold

In [ ]:
best_threshold = joblib.load('../models/best_threshold.pkl')
y_val_pred = (y_val_pred_prob >= best_threshold).astype(int)
y_test_pred = (y_test_pred_prob >= best_threshold).astype(int)

### 3. Precision, Recall, F1, F2 (on delayed class specifically)

In [24]:
round(accuracy_score(y_test, y_test_pred) * 100, 2)

80.08

In [14]:
precision = precision_score(y_test, y_test_pred, average='binary')
recall = recall_score(y_test, y_test_pred, average='binary')
f1 = f1_score(y_test, y_test_pred, average='binary')
f"Precision: {round(precision * 100, 2)}%", f"Recall: {round(recall * 100, 2)}%", f"F1-Score: {round(f1 * 100, 2)}%"

('Precision: 53.36%', 'Recall: 33.71%', 'F1-Score: 41.32%')

### 4. Confusion Matrix

In [15]:
print(confusion_matrix(y_test, y_test_pred))

[[993420  83341]
 [187511  95356]]


In [16]:
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.84      0.92      0.88   1076761
           1       0.53      0.34      0.41    282867

    accuracy                           0.80   1359628
   macro avg       0.69      0.63      0.65   1359628
weighted avg       0.78      0.80      0.78   1359628

